In [3]:
import numpy as np
from tqdm import tqdm
import cv2
import os
import imutils


def crop_img(img):
    """
    Finds the extreme points on the image and crops the rectangular out of them.
    """
    # Convert to grayscale for contour detection
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    gray = cv2.GaussianBlur(gray, (3, 3), 0)

    # Threshold then perform erosions and dilations
    thresh = cv2.threshold(gray, 45, 255, cv2.THRESH_BINARY)[1]
    thresh = cv2.erode(thresh, None, iterations=2)
    thresh = cv2.dilate(thresh, None, iterations=2)

    # Find contours and get largest one
    cnts = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cnts = imutils.grab_contours(cnts)
    if len(cnts) == 0:
        return img  # Return original if no contours found
    c = max(cnts, key=cv2.contourArea)

    # Find extreme points
    extLeft = tuple(c[c[:, :, 0].argmin()][0])
    extRight = tuple(c[c[:, :, 0].argmax()][0])
    extTop = tuple(c[c[:, :, 1].argmin()][0])
    extBot = tuple(c[c[:, :, 1].argmax()][0])
    
    ADD_PIXELS = 0
    new_img = img[extTop[1]-ADD_PIXELS:extBot[1]+ADD_PIXELS, extLeft[0]-ADD_PIXELS:extRight[0]+ADD_PIXELS].copy()

    return new_img


def process(root_source, root_dest):
    IMG_SIZE = 224  # ResNet standard input size
    for dir in os.listdir(root_source):
        class_path = os.path.join(root_source, dir)
        if not os.path.isdir(class_path):
            continue
        save_path = os.path.join(root_dest, dir)
        if not os.path.exists(save_path):
            os.makedirs(save_path)
        image_names = os.listdir(class_path)
        for img_name in tqdm(image_names, desc=f"Processing {dir}"):
            img_path = os.path.join(class_path, img_name)
            image = cv2.imread(img_path)
            if image is None:
                print(f"Skipping unreadable image: {img_path}")
                continue
            # Convert BGR to RGB (PyTorch expects RGB)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            new_img = crop_img(image)
            new_img = cv2.resize(new_img, (IMG_SIZE, IMG_SIZE))
            # Convert back RGB to BGR for saving if needed (optional)
            new_img = cv2.cvtColor(new_img, cv2.COLOR_RGB2BGR)
            out_path = os.path.join(save_path, img_name)
            cv2.imwrite(out_path, new_img)


if __name__ == "__main__":
    training = "/Users/ishabhansali/Downloads/federated_learning_new/Training"
    testing = "/Users/ishabhansali/Downloads/federated_learning_new/Testing"
    cleaned_training = "/Users/ishabhansali/Downloads/federated_learning_new/cleaned/Training"
    cleaned_testing = "/Users/ishabhansali/Downloads/federated_learning_new/cleaned/Testing"
    
    process(training, cleaned_training)
    process(testing, cleaned_testing)


Processing meningioma: 100%|██████████| 306/306 [00:00<00:00, 975.06it/s]
